[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/grok/examples-composite_scoring/examples/composite_scoring/composite_scoring.ipynb)

# Combine atomic scores with your own weights

"Is this applicant worth a call?" mixes the stack, the writing and the tone. One score would bake the mixture into the question, and a new priority would mean a new prompt. FRIDA-Decisions scores each factor on its own scale. The weights, and the cutoff that means "invite", are ordinary numbers.

Pattern: [Composite scoring](https://docs.typesafe.ai/patterns/composite-scoring). The letters are synthetic and short; the accuracy on `data.json` is a check that the cutoff does something sensible, not a hiring metric.


In [1]:
import sys
!{sys.executable} -m pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


In [2]:
import json, os
from pathlib import Path
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

frida-decisions 0.4.0, loaded on cpu, torch.float32


In [3]:
ROLE = "Вакансия: бэкенд на Python, нужны Django, PostgreSQL и опыт разбора чужих инцидентов."

RUBRICS = {
    "skills": [
        "Опыта по делу нет.",
        "Смежный опыт, нужного стека почти нет.",
        "Часть стека есть, без конкретных задач.",
        "Нужный стек и названы конкретные задачи.",
        "Стек, цифры и самостоятельные решения.",
    ],
    "clarity": [
        "Непонятно, что человек умеет.",
        "Общие слова, почти без фактов.",
        "Факты есть, но их приходится вычитывать.",
        "Письмо по делу и его легко дочитать.",
        "Коротко, по делу, с примерами.",
    ],
    "tone": [
        "Грубо или пренебрежительно.",
        "Фамильярно, рабочему письму не подходит.",
        "Нейтрально, без явных сбоев.",
        "Спокойно и уважительно.",
        "Уважительно и без заискивания.",
    ],
}

WEIGHTS = {
    "stack": {"skills": 0.7, "clarity": 0.2, "tone": 0.1},
    "writing": {"skills": 0.2, "clarity": 0.5, "tone": 0.3},
}
# `score` is the expected level, 0 .. 4. On this rubric the letters land in the
# lower half of the scale, so the invite cutoff is 1.15, not the midpoint.
CUTOFF = 1.15


def request_for(text: str) -> dict:
    return {
        "state": ROLE + "\n\nПисьмо кандидата:\n" + text,
        "questions": {
            "skills": {"type": "score", "instructions": "Насколько опыт совпадает с вакансией?",
                       "criteria": RUBRICS["skills"]},
            "clarity": {"type": "score", "instructions": "Насколько письмо ясное и конкретное?",
                        "criteria": RUBRICS["clarity"]},
            "tone": {"type": "score", "instructions": "Насколько тон уместен для рабочего письма?",
                     "criteria": RUBRICS["tone"]},
        },
    }


def weighted(answers: dict, weights: dict) -> float:
    return sum(weights[name] * answers[name]["score"] for name in weights)


## Change a coefficient, not a prompt

Both columns use the same three answers. Only the weights change, and with them who clears the cutoff.


In [4]:
def run(judge, rows: list[dict]) -> None:
    scored = []
    for row in rows:
        response = judge(request_for(row["text"]))
        answers = response["answers"]
        values = {name: weighted(answers, weights) for name, weights in WEIGHTS.items()}
        scored.append({**row, "answers": answers, "values": values})
        if response["usage"]["state_truncated"]:
            print(f"  {row['name']}: state truncated")

    print(f"{'name':10} {'skills':6} {'clarity':7} {'tone':6} {'stack':7} {'writing':7} invite")
    hits = 0
    for row in scored:
        answers = row["answers"]
        invite = row["values"]["stack"] >= CUTOFF
        hits += invite == row["invite"]
        print(f"{row['name']:10} {answers['skills']['score']:.2f}   {answers['clarity']['score']:.2f}    "
              f"{answers['tone']['score']:.2f}   {row['values']['stack']:.2f}    "
              f"{row['values']['writing']:.2f}    {str(invite):5} gold {row['invite']}")

    by_stack = [row["name"] for row in sorted(scored, key=lambda row: -row["values"]["stack"])]
    by_writing = [row["name"] for row in sorted(scored, key=lambda row: -row["values"]["writing"])]
    print(f"\norder by stack weights:   {', '.join(by_stack)}")
    print(f"order by writing weights: {', '.join(by_writing)}")
    print(f"cutoff {CUTOFF} on the stack weights agrees with the invite label in {hits}/{len(rows)}")


In [5]:
def load_rows():
    import urllib.request
    path = Path("data.json")
    if not path.exists():
        url = ("https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/"
               "grok/examples-composite-scoring/examples/composite_scoring/data.json")
        path.write_bytes(urllib.request.urlopen(url, timeout=60).read())
    return json.loads(path.read_text(encoding="utf-8"))

run(judge, load_rows())


name       skills clarity tone   stack   writing invite
Ира        1.41   0.94    3.05   1.48    1.67    True  gold True
Олег       1.37   1.54    3.07   1.58    1.97    True  gold True
Маша       0.80   0.13    1.02   0.69    0.53    False gold False
Кирилл     0.85   0.24    1.52   0.79    0.75    False gold False
Нина       0.72   0.27    1.23   0.68    0.65    False gold False
Павел      1.32   0.72    2.19   1.29    1.28    True  gold True

order by stack weights:   Олег, Ира, Павел, Кирилл, Маша, Нина
order by writing weights: Олег, Ира, Павел, Кирилл, Нина, Маша
cutoff 1.15 on the stack weights agrees with the invite label in 6/6
